<a href="https://colab.research.google.com/github/vulpecriptografica/Symm4ML/blob/main/lie_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lie Groups — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Lie Groups exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/lie_companion.ipynb)

## Setup

In [36]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [37]:
import numpy as np
from numpy import einsum as ein
from typing import List
import scipy

from symm4ml import groups, linalg, rep, lie

### Reference data

These structure constants and generators are used throughout the exercise for testing.

In [38]:
# SO(3) structure constants
so3_A = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) L=1 generators
so3_X = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(1,3) structure constants
so13_A = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
        ],
        [
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SO(1,3) generators
so13_X = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 1.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SU(2) generators
su2_X = np.array(
    [
        [
            [0.0 + 0.0j, 0.5 + 0.0j],
            [-0.5 + 0.0j, 0.0 + 0.0j],
        ],
        [
            [-0.0 - 0.5j, 0.0 + 0.0j],
            [0.0 + 0.0j, 0.0 + 0.5j],
        ],
        [
            [0.0 - 0.0j, 0.0 + 0.5j],
            [0.0 + 0.5j, 0.0 - 0.0j],
        ],
    ]
)

print(f"so3_A: {so3_A.shape}, so3_X: {so3_X.shape}")
print(f"so13_A: {so13_A.shape}, so13_X: {so13_X.shape}")
print(f"su2_X: {su2_X.shape}")

so3_A: (3, 3, 3), so3_X: (3, 3, 3)
so13_A: (6, 6, 6), so13_X: (6, 4, 4)
su2_X: (3, 2, 2)


---
## 1. `are_isomorphic(X1, X2)`

Check if two representations of a Lie algebra (expressed in terms of generators) are isomorphic, i.e., the same up to a similarity transform. You can use a function from a previous problem set.

In [39]:
def are_isomorphic(X1: np.ndarray, X2: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if two representations of a Lie group are isomorphic.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        are_isomorphic: bool - True if the representations are isomorphic.
    """
    e_x1 = scipy.linalg.expm(X1)
    e_x2 = scipy.linalg.expm(X1)
    return rep.are_isomorphic(e_x1, e_x2)

In [40]:
# Small tests from lie.py
assert are_isomorphic(so3_X, so3_X)
print("are_isomorphic tests passed!")

are_isomorphic tests passed!


---
## 2. `tensor_product(X1, X2)`

Compute the tensor product of two representations of a Lie algebra, where both input and output are expressed in terms of generators.

In [41]:
def tensor_product(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    """Tensor product of two representations of a Lie group.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        X: np.array [lie_dim, d1*d2, d1*d2] - tensor product of the representations.
    """
    d1 = X1.shape[1]
    d2 = X2.shape[1]
    X1_prom = np.kron(X1, np.eye(d2))
    X2_prom = np.kron(np.eye(d1), X2)
    return X1_prom + X2_prom

In [42]:
# Small tests from lie.py
assert tensor_product(so3_X, so3_X).shape == (3, 9, 9)
print("tensor_product tests passed!")

tensor_product tests passed!


---
## 3. `is_a_representation(algebra, X)`

Check whether the given generators `X` satisfy the commutation relations encoded in `algebra`: $[X_i, X_j] = \sum_k A_{ijk} X_k$.

In [43]:
def is_a_representation(
    algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8
) -> bool:
    """Check if X satisfies the commutation relations of the Lie algebra:
        [X_i, X_j] = sum_k A_{ijk} X_k
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_a_representation: bool - True if X satisfies the commutation relations.
    """
    """Check if X satisfies the commutation relations of the Lie algebra:
        [X_i, X_j] = sum_k A_{ijk} X_k
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_a_representation: bool - True if X satisfies the commutation relations.
    """
    num_gen = X.shape[0]
    for i in range(num_gen):
      for j in range(num_gen):
        lhs = X[i]@ X[j] - X[j]@ X[i]
        A_ij = algebra[i][j]
        print(A_ij)
        rhs = np.einsum('a, abc-> bc', A_ij, X)
        if np.allclose(lhs, rhs):
          continue
        else:
          return False
    return True

In [44]:
# Small tests from lie.py
assert is_a_representation(so3_A, so3_X)
print("is_a_representation tests passed!")

[0. 0. 0.]
[0. 0. 1.]
[ 0. -1.  0.]
[ 0.  0. -1.]
[0. 0. 0.]
[1. 0. 0.]
[0. 1. 0.]
[-1.  0.  0.]
[0. 0. 0.]
is_a_representation tests passed!


---
## 4. `is_an_irrep(algebra, X)`

Check if a representation of a Lie algebra is irreducible. Return `True` if the input is a valid representation AND is irreducible.

In [45]:
def is_an_irrep(algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if X is an irreducible representation of the Lie algebra.
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_an_irrep: bool - True if X is an irreducible representation.
    """
    # apply schur's lemma to find irreps: irreps iff U = cI for const c
    U = linalg.infer_change_of_basis(X, X)
    dim_U = U.shape[1]
    c = U[0][0][0] # scaling constant
    return np.allclose(U, c*np.eye(dim_U), tol)

In [46]:
# Small tests from lie.py
assert is_an_irrep(so3_A, so3_X)
print("is_an_irrep tests passed!")

is_an_irrep tests passed!


---
## 5. `decompose_rep_into_irreps(X)`

Decompose a representation of a Lie algebra into a direct sum of irreducible representations. Follow the same algorithm as for finite groups.

In [47]:
def decompose_rep_into_irreps(X: np.array, *, tol: float = 1e-8) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        Ys: List[np.array] - list of generators of irreducible representations.
    """
    # step 1: find change of basis Qs
    Qs = linalg.infer_change_of_basis(X, X)

    # step 2: random lin comb to break accidental degeneracies
    # this helps us get different eigenspaces -> diff similarity transforms B
    d = len(Qs)
    a = np.random.rand(d)
    Q_ = np.einsum('i,ijk->jk', a, Qs)

    # step 3: find eigenspaces of Q_ (one-to-one w/ irreps)
    evals, evecs = np.linalg.eig(Q_)
    espaces = linalg.eigenspaces(evals, evecs) # d x n for evec matrices

    # step 4: orthonormalize and extract irreps
    irreps = []
    l = len(espaces)
    for i in range(l): # for each espace
      B = espaces[i][1] # arr of evecs
      irr = np.einsum('ia, hij, jb -> hab' , B, X, B.conj())
      irreps.append(irr)
    return irreps

ans = {(lie.is_an_irrep(so3_A, X), X.shape) for X in decompose_rep_into_irreps(tensor_product(so3_X, so3_X))}
print(ans)


{(False, (3, 3, 3)), (True, (3, 1, 1)), (False, (3, 5, 5))}


In [48]:
linalg.infer_change_of_basis.__doc__

'Compute the change of basis matrix from X1 to X2.\n    tip: Use the function nullspace\n    Input:\n        X1: an (n, d1, d1) array of n (d1, d1) matrices\n        X2: an (n, d2, d2) array of n (d2, d2) matrices\n    Output:\n        Sols: An (m, d1, d2) array of m solutions.\n        Each solution is a (d1, d2) matrix that satisfies X1 @ S = S @ X2,\n        and together they form an orthognal basis for the set of solutions (under the inner product of the flattened versions).\n    '

In [49]:
# Small tests from lie.py
assert {
    X.shape[1] for X in decompose_rep_into_irreps(tensor_product(so3_X, so3_X))
} == {1, 3, 5}
print("decompose_rep_into_irreps tests passed!")

decompose_rep_into_irreps tests passed!


---
## 6. `infer_irreps_from_tensor_products(X, n)`

Infer `n` non-isomorphic finite-dimensional irreps of the underlying matrix Lie group by successively decomposing tensor products. Start with the trivial representation and take successive tensor products with `X` and (if needed) its conjugate `X*`.

In [82]:
def infer_irreps_from_tensor_products(
    X: np.ndarray, n: int, *, tol: float = 1e-8
) -> List[np.ndarray]:
    """Infers irreducible representations from successive tensor products of a representation.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
        n: int - number of non-isomorphic irreducible representations to infer.
    Output:
        Ys: List[np.array] - list of n generators of irreducible representations,
            each an np.array of shape [lie_dim, d', d'] for some d'.
    """

    # initializing
    irreps = []

    # start by doing 0 + X + X*
    lie_dim = X.shape[0]
    zero = np.array([np.zeros((1,1)) for i in range(lie_dim)])
    first = rep.direct_sum(zero, X)
    big_X = rep.direct_sum(first, X.conj())
    tp_now = big_X


    # getting more irreps
    while len(irreps) < n:
      # decomp
      new_irs = lie.decompose_rep_into_irreps(tp_now)
      if not irreps:
        idxs = []
        for i in range(len(new_irs)-1):
          if any(lie.are_isomorphic(arr,new_irs[i]) for arr in new_irs[i+1:]):
            idxs.append(i)
        for i in idxs:
          new_irs.pop(i)
        irreps += new_irs
      else:
        idxs = []
        for i in range(len(new_irs)):
          if any(lie.are_isomorphic(arr,new_irs[i]) for arr in irreps):
            idxs.append(i)
        for i in idxs:
          print('this is new_irs currently', new_irs)
          print('these are the indices we want to pop', idxs)
          new_irs.pop(i)
        irreps += new_irs
      print('cirrent len of irreps:', len(irreps))
      tp_now = lie.tensor_product(tp_now, big_X)

      # # check if new irreps are isomorphic to prev irreps found
      # if not irreps:
      #   for i in new_irs:
      #     for j in new_irs:
      #       if not lie.are_isomorphic(i,j):
      #         irreps.append(i)
      #         print('girlll')
      #       else:
      #         print('isomorphic booo')
      # else:
      #   for i in new_irs:
      #     for j in irreps:
      #       if not lie.are_isomorphic(i,j):
      #         irreps.append(i)
      #         print('hii')
      # break
      # if len(irreps) == n:
      #   break
      # tp_now = lie.tensor_product(tp_now, big_X)

    print(irreps)

    return irreps




    # # start by getting irreps of generators X
    # og_irs = lie.decompose_rep_into_irreps(X)

    # # also get trivial scalar irrep 0
    # d = X.shape[1]
    # zero = np.zeros((1,1))  # smallest irrep

    # # initialize lists for keeping track of irreps
    # p_todo = [zero] + [i for i in og_irs]
    # p_done = [zero] + [i for i in og_irs]

    # # get more irreps, starting by tensor producting w/ og_ir elts
    # while len(p_done) != n:
    #   ir_now = p_todo.pop(0)
    #   # tensor prod on generators
    #   new = tensor_product(ir_now, )




    # # start w/ trivial rep
    # d = X.shape[1]
    # first_two = rep.direct_sum(X, X.conj())
    # X_start = rep.direct_sum(np.zeros(d,d), first_two)

    # # get irreps of this
    # ir_start = lie.decompose_rep_into_irreps(X_start)



Xs = infer_irreps_from_tensor_products(so3_X, 3)

cirrent len of irreps: 2
this is new_irs currently [array([[[ 0.00000000e+00+0.j]],

       [[ 7.70371978e-34+0.j]],

       [[-1.12869875e-33+0.j]]]), array([[[-1.23259516e-32+0.j]],

       [[ 0.00000000e+00+0.j]],

       [[-4.16217343e-33+0.j]]]), array([[[ 1.23259516e-32+0.j]],

       [[ 0.00000000e+00+0.j]],

       [[-5.36850447e-33+0.j]]]), array([[[-1.03292366e-32+0.j, -3.28399826e-16+0.j,  1.01803978e-15+0.j],
        [ 3.28399826e-16+0.j, -4.15681018e-33+0.j, -1.00000000e+00+0.j],
        [-1.01803978e-15+0.j,  1.00000000e+00+0.j,  8.66974047e-33+0.j]],

       [[-3.05143244e-32+0.j, -3.22312094e-16+0.j,  1.00000000e+00+0.j],
        [ 3.22312094e-16+0.j, -8.91432204e-34+0.j,  1.01803978e-15+0.j],
        [-1.00000000e+00+0.j, -1.01803978e-15+0.j,  7.05554431e-33+0.j]],

       [[-1.14041020e-32+0.j, -1.00000000e+00+0.j, -3.22312094e-16+0.j],
        [ 1.00000000e+00+0.j, -2.20142731e-32+0.j,  3.28399826e-16+0.j],
        [ 3.22312094e-16+0.j, -3.28399826e-16+0.j, -5.627799

IndexError: pop index out of range

In [63]:
# Small tests from lie.py
Xs = infer_irreps_from_tensor_products(so3_X, 3)
assert len(Xs) == 3
assert Xs[0].shape == (3, 1, 1)
assert is_an_irrep(so3_A, Xs[0])
assert Xs[1].shape == (3, 3, 3)
assert is_an_irrep(so3_A, Xs[1])
assert Xs[2].shape == (3, 5, 5)
assert is_an_irrep(so3_A, Xs[2])
print("infer_irreps_from_tensor_products tests passed!")

this is the current number of irreps:, 3
isomorphic booo
isomorphic booo
girlll
isomorphic booo
isomorphic booo
girlll
girlll
girlll
isomorphic booo
[array([[[ 0.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0.,  0.,  0.],
        [-1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0.,  0.]]]), array([[[ 0.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0.,  0.,  0.],
        [-1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0.,  0.]]]), array([[[0.]],

       [[0.]],

       [[0.]]]), array([[[0.]],

       [[0.]],

       [[0.]]])]


AssertionError: 

In [64]:
dir(rep)
rep.direct_sum.__doc__

'Computes direct sum of two representations.\n    Input:\n        rep1: np.array [n, d1, d1] representation of group. rep[i] is a matrix that\n            represents i-th element of group.\n        rep2: np.array [n, d2, d2] representation of group. rep[i] is a matrix that\n            represents i-th element of group.\n        You can assume that rep1 and rep2 are valid group representations.\n    Output:\n        Direct sum of representations. np.array [n, d1 + d2, d1 + d2].\n    '

---
## Explore Further

In [ ]:
# Try inferring irreps of SO(1,3) or SU(2)!